# Drive manifest reload smoke

该 Notebook 用于 Colab 冷启动后的复核: 挂载 Google Drive, 拉取仓库代码, 只根据 Drive manifest 重载并校验已镜像文件。

正式逻辑位于 `paper_workflow/colab_utils/drive_workflow.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 先运行 `colab_drive_cold_start_smoke.ipynb` 或等价命令生成 Drive manifest。
2. 默认 Drive 根目录为 `/content/drive/MyDrive/SLM`。
3. 该 Notebook 不扫描未登记文件, 只读取 manifest 中列出的镜像文件。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_DRIVE_ROOT', '/content/drive/MyDrive/SLM')
os.environ.setdefault('SLM_WM_WORKFLOW_NAME', 'colab_drive_workflow')
print({
    'drive_root': os.environ['SLM_WM_DRIVE_ROOT'],
    'workflow_name': os.environ['SLM_WM_WORKFLOW_NAME'],
})


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

from paper_workflow.colab_utils.drive_workflow import write_reload_smoke_record

reload_record = write_reload_smoke_record(
    root='.',
    drive_root=os.environ.get('SLM_WM_DRIVE_ROOT', '/content/drive/MyDrive/SLM'),
    workflow_name=os.environ.get('SLM_WM_WORKFLOW_NAME', 'colab_drive_workflow'),
)
reload_record


In [ ]:
from pathlib import Path

reload_path = Path('outputs/colab_drive_workflow/reload_smoke_record.jsonl')
print(reload_path)
print(reload_path.read_text(encoding='utf-8'))
